<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur la rue de la convention.

Imports

In [84]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'convention.csv'

df_final = pd.read_csv('../datasets_axes_with_all_features/'+ doc, sep=';')


In [85]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9266 entries, 0 to 9265
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 9266 non-null   int64  
 1   Identifiant arc            9266 non-null   int64  
 2   Libelle                    9266 non-null   object 
 3   Date et heure de comptage  9266 non-null   object 
 4   Débit horaire              4685 non-null   float64
 5   Taux d'occupation          4792 non-null   float64
 6   Etat trafic                9266 non-null   object 
 7   Identifiant noeud amont    9266 non-null   int64  
 8   Libelle noeud amont        9266 non-null   object 
 9   Identifiant noeud aval     9266 non-null   int64  
 10  Libelle noeud aval         9266 non-null   object 
 11  Etat arc                   9266 non-null   object 
 12  Date debut dispo data      9266 non-null   object 
 13  Date fin dispo data        9266 non-null   objec

In [86]:
df_final.describe()

,Unnamed: 0,Identifiant arc,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,duree prec (en min),force moyenne vent (m/s),Température,ensoleillement (en min),est_pieton,est_vacances,est_avant_vacances,est_ferie,est_avant_ferie,est_weekend
count,9266.000000,9266.0,4685.000000,4792.000000,9266.0,9266.0,9266.000000,9266.000000,9266.000000,9266.000000,...,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000,9266.000000
mean,4632.500000,5671.0,398.531057,3.590080,2937.0,2973.0,11.505720,2.991150,6.762789,2024.738291,...,4.022232,2.938431,13.447151,13.021261,0.012627,0.328405,0.015649,0.025901,0.025901,0.283510
std,2675.008131,0.0,225.331508,3.744822,0.0,0.0,6.920146,2.002354,3.345876,0.439589,...,12.929655,1.314630,6.599812,21.858906,0.111663,0.469658,0.124118,0.158849,0.158849,0.450726
min,0.000000,5671.0,31.000000,0.000000,2937.0,2973.0,0.000000,0.000000,1.000000,2024.000000,...,0.000000,0.000000,-3.600000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2316.250000,5671.0,196.000000,1.310973,2937.0,2973.0,6.000000,1.000000,4.000000,2024.000000,...,0.000000,2.000000,9.100000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4632.500000,5671.0,389.000000,2.623890,2937.0,2973.0,12.000000,3.000000,7.000000,2025.000000,...,0.000000,2.800000,13.400000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,6948.750000,5671.0,578.000000,4.442778,2937.0,2973.0,17.000000,5.000000,10.000000,2025.000000,...,0.000000,3.700000,17.800000,20.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,9265.000000,5671.0,1024.000000,37.348340,2937.0,2973.0,23.000000,6.000000,12.000000,2025.000000,...,60.000000,9.300000,37.600000,60.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [87]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'est_vacances', 'duree prec (en min)',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'jour_semaine', 'mois',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'est_vacances',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'duree prec (en min)', 'mois',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=8,
    max_iter=400,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
W_VAC = 5
POST_SCALE_FERIE = 1.0  # laissez 1.0 si vous ne souhaitez pas corriger les prédictions les jours fériés

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()
    is_vacances = X_frame['est_vacances'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON
    w[is_vacances == 1] = W_VAC

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

# Diagnostics par sous-régimes
est_vacances_test = X_test['est_vacances'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if est_vacances_test.any():
    mae_vac = mean_absolute_error(y_test[est_vacances_test], y_pred[est_vacances_test])
    print(f"MAE (jours vacances) : {mae_vac:.2f} (n={est_vacances_test.sum()})")
    mae_non_vac = mean_absolute_error(y_test[~est_vacances_test], y_pred[~est_vacances_test])
    print(f"MAE (jours non vacances) : {mae_non_vac:.2f} (n={(~est_vacances_test).sum()})")

# ---------- 8) Imputation des valeurs manquantes ----------
if mask_missing.any():
    df_final.loc[mask_missing, 'Débit_prédit'] = pipe.predict(X_missing)
    print(f"Imputation réalisée pour {mask_missing.sum()} lignes (colonne 'Débit_prédit').")


R² : 0.902
MAE : 48.54
RMSE : 68.69
Part d'observations piéton (test) : 2.8%
MAE (jours piéton) : 35.62 (n=26)
MAE (jours non piéton) : 48.91 (n=911)
Part d'observations piéton (test) : 2.8%
MAE (jours vacances) : 61.70 (n=289)
MAE (jours non vacances) : 42.68 (n=648)
Imputation réalisée pour 4581 lignes (colonne 'Débit_prédit').


In [90]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
3                  heure_sin       176.217322        4.049799
4                  heure_cos       116.627027        4.057349
5                   jour_sin        21.006595        1.204081
1               est_vacances         5.752326        1.210310
10              jour_semaine         4.413143        0.468180
6                   jour_cos         1.514817        0.282573
9   force moyenne vent (m/s)         1.078510        0.396514
14   ensoleillement (en min)         0.555518        0.259514
2        duree prec (en min)         0.543657        0.467842
0                Température         0.362875        0.225584
15                est_pieton         0.019013        0.014333
11                      mois         0.000000        0.000000
12                 est_ferie         0.000000        0.000000
13           est_avant_ferie         0.000000        0.000000
8                   mois_cos        -0.013342        0.003980
7       

In [91]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()